In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kushalag21/iemocap-denoised-dataset/B2/iemocap_audio_features_denoised.csv
/kaggle/input/datasets/kushalag21/iemocap-denoised-dataset/B2/processed_iemocap_audio_features_5.csv
/kaggle/input/datasets/kushalag21/iemocap-denoised-dataset/SpeechCueLLM/README.md
/kaggle/input/datasets/kushalag21/iemocap-denoised-dataset/SpeechCueLLM/feature_distributions_gender_F.png
/kaggle/input/datasets/kushalag21/iemocap-denoised-dataset/SpeechCueLLM/feature_distributions_gender_M.png
/kaggle/input/datasets/kushalag21/iemocap-denoised-dataset/SpeechCueLLM/feature_distributions_overall.png
/kaggle/input/datasets/kushalag21/iemocap-denoised-dataset/SpeechCueLLM/image.png
/kaggle/input/datasets/kushalag21/iemocap-denoised-dataset/SpeechCueLLM/.git/HEAD
/kaggle/input/datasets/kushalag21/iemocap-denoised-dataset/SpeechCueLLM/.git/config
/kaggle/input/datasets/kushalag21/iemocap-denoised-dataset/SpeechCueLLM/.git/description
/kaggle/input/datasets/kushalag21/iemocap-denoised-dataset/Spe

In [2]:
import time; T0 = time.time()
!pip install -q deepspeed bitsandbytes peft opensmile
!pip uninstall -y -q torchao

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import os, re, json, glob, shutil, subprocess
!git clone -q https://github.com/zehuiwu/SpeechCueLLM /kaggle/working/SpeechCueLLM
os.chdir("/kaggle/working/SpeechCueLLM/LLM_code")

In [4]:
hits = glob.glob("/kaggle/input/**/processed_iemocap_audio_features_5.csv", recursive=True)
print(hits)                      # expect exactly one path
B2 = hits[0]
DST = "/kaggle/working/SpeechCueLLM/speech_features/processed_iemocap_audio_features_5.csv"
shutil.copy(DST, "/kaggle/working/processed_iemocap_RAW_backup.csv")
shutil.copy(B2, DST)

import pandas as pd
d = pd.read_csv(DST); print(d.shape)
print(d['description'].iloc[0])

['/kaggle/input/datasets/kushalag21/iemocap-denoised-dataset/B2/processed_iemocap_audio_features_5.csv', '/kaggle/input/datasets/kushalag21/iemocap-denoised-dataset/SpeechCueLLM/speech_features/processed_iemocap_audio_features_5.csv']
(10039, 25)
Target speech characteristics: moderate volume with moderate variation, very low pitch with very low variation, high speaking rate.


In [5]:
!rm -rf /kaggle/working/llama2-7b-hf
!pip install -q huggingface_hub
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

REPO = "meta-llama/Llama-2-7b-hf"
TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.makedirs("/kaggle/temp", exist_ok=True)

MODEL_PATH = snapshot_download(
    REPO,
    local_dir="/kaggle/temp/llama2-7b-hf",
    allow_patterns=["*.json", "*.safetensors", "tokenizer.model"],
    token=TOKEN,
)
EPOCHS = 3         
DATASET, LR = "iemocap", "3e-4"
assert os.path.exists(MODEL_PATH + "/config.json")
print(MODEL_PATH, os.listdir(MODEL_PATH))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

/kaggle/temp/llama2-7b-hf ['pytorch_model.bin.index.json', 'generation_config.json', 'special_tokens_map.json', 'config.json', 'tokenizer.json', 'tokenizer_config.json', 'tokenizer.model', 'model-00001-of-00002.safetensors', '.cache', 'model-00002-of-00002.safetensors', 'model.safetensors.index.json']


In [6]:
s = open("main.py").read()
s = s.replace(
"        model =AutoModelForCausalLM.from_pretrained(args.model_name_or_path).half()",
"""        import time, gc, torch as _t
        if int(os.environ.get("LOCAL_RANK", 0)) == 1:
            time.sleep(180)
        from transformers import BitsAndBytesConfig
        _bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                  bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=_t.float16)
        model = AutoModelForCausalLM.from_pretrained(args.model_name_or_path, quantization_config=_bnb,
                    torch_dtype=_t.float16, low_cpu_mem_usage=True,
                    device_map={"": int(os.environ.get("LOCAL_RANK", 0))})
        gc.collect()
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=bool(args.gradient_checkpointing))""")
s = s.replace('deepspeed_config["bfloat16"]["enabled"] = True\n        deepspeed_config["fp16"]["enabled"] = False',
              'deepspeed_config["bfloat16"]["enabled"] = False\n        deepspeed_config["fp16"]["enabled"] = True')
s = s.replace("    return optimizer_grouped_parameters",
              "    return [g for g in optimizer_grouped_parameters if len(g['params']) > 0]")
s = s.replace("    deepspeed_config[\"zero_optimization\"]['mics_shard_size'] = world_size", "    pass")
open("main.py", "w").write(s)

p = "data_utils/deepspeed_config.json"
cfg = json.load(open(p)); cfg.pop("amp", None); cfg["zero_optimization"]["stage"] = 2
json.dump(cfg, open(p, "w"), indent=2)
print("4bit", "BitsAndBytesConfig" in s, "| fp16", 'deepspeed_config["fp16"]["enabled"] = True' in s, "| amp removed", "amp" not in cfg)

4bit True | fp16 True | amp removed True


In [7]:
sh = open("train_and_inference.sh").read()
def setv(name, val):
    global sh; sh = re.sub(rf"^{name}=.*$", f"{name}={val}", sh, count=1, flags=re.M)
setv("MODEL_NAME", "'LLaMA2'"); setv("Experiments_setting", "'lora'"); setv("dataset", f"'{DATASET}'")
setv("audio_description", "'True'"); setv("audio_context", "'True'")
setv("audio_impression", "'False'"); setv("audio_only", "'False'")
setv("SEED", 11); setv("num_train_epochs", EPOCHS); setv("LORA_LR", LR)
setv("accumulations", 4); setv("graphics_card", 2); setv("data_percent", 1.0)
sh = sh.replace("MODEL_PATH='LLaMA2 MODELPATH'", f"MODEL_PATH='{MODEL_PATH}'")
sh = sh.replace("--seed ${SEED}\n    fi", "--seed ${SEED} \\\n        --gradient_checkpointing\n    fi")
open("train_and_inference.sh", "w").write(sh)
!grep -n -E "^(audio_|dataset|num_train|data_percent|SEED)" train_and_inference.sh

21:dataset='iemocap'
25:audio_description='True'
26:audio_impression='False'
27:audio_context='True'
28:audio_only='False'
31:SEED=11
32:num_train_epochs=3
55:data_percent=1.0


In [8]:
LOG = "/kaggle/working/run_log_B2_denoised_7B.txt"
proc = subprocess.Popen(f"PYTHONUNBUFFERED=1 bash train_and_inference.sh > {LOG} 2>&1", shell=True)
print("started, pid", proc.pid)

started, pid 328


In [9]:
DEADLINE_H = 11.0

def status():
    lines = open(LOG, errors="ignore").read().replace("\r", "\n").split("\n")
    steps = [l for l in lines if "Running Loss" in l]
    last = steps[-1].strip()[-140:] if steps else "(loading model / not started)"
    f1 = [l for l in lines if l.startswith(("current_f1", "best_f1", "history"))][-3:]
    err = [l for l in lines if "Traceback" in l or "OutOfMemory" in l or "Killed" in l][-2:]
    gpu = subprocess.run("nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader",
                         shell=True, capture_output=True, text=True).stdout.strip().replace("\n", " | ")
    ram = subprocess.run("free -g | awk 'NR==2{print $3\"/\"$2\" GB\"}'", shell=True, capture_output=True, text=True).stdout.strip()
    print(f"[{time.strftime('%H:%M:%S')} +{(time.time()-T0)/3600:.1f}h] {last}\n  GPU: {gpu} | RAM: {ram}")
    for l in f1 + err: print("  ", l.strip()[:200])

while proc.poll() is None:
    status(); time.sleep(300)
    if time.time() - T0 > DEADLINE_H * 3600:
        print("DEADLINE reached, stopping training"); subprocess.run("pkill -f main.py", shell=True); break
status()
subprocess.run("pkill -f main.py; pkill -f deepspeed", shell=True)
print("FINISHED, exit code", proc.poll())

[09:03:00 +0.0h] (loading model / not started)
  GPU: 0 MiB, 0 % | 0 MiB, 0 % | RAM: 2/31 GB
[09:08:00 +0.1h] (loading model / not started)
  GPU: 6425 MiB, 0 % | 3 MiB, 0 % | RAM: 4/31 GB
[09:13:00 +0.2h] Epochs 0/3. Running Loss:    0.7954:   6%|▋         | 165/2582 [04:09<59:22,  1.47s/it]
  GPU: 5331 MiB, 100 % | 5227 MiB, 100 % | RAM: 7/31 GB
[09:18:00 +0.3h] Epochs 0/3. Running Loss:    0.0536:  14%|█▎        | 355/2582 [09:10<59:13,  1.60s/it]
  GPU: 5417 MiB, 93 % | 5227 MiB, 95 % | RAM: 7/31 GB
[09:23:00 +0.4h] Epochs 0/3. Running Loss:    0.7972:  21%|██        | 543/2582 [14:09<59:29,  1.75s/it]
  GPU: 5417 MiB, 100 % | 5227 MiB, 100 % | RAM: 7/31 GB
[09:28:00 +0.4h] Epochs 0/3. Running Loss:    0.2419:  28%|██▊       | 731/2582 [19:10<46:18,  1.50s/it]
  GPU: 5417 MiB, 100 % | 5227 MiB, 100 % | RAM: 7/31 GB
[09:33:00 +0.5h] Epochs 0/3. Running Loss:    0.1893:  36%|███▌      | 919/2582 [24:09<41:22,  1.49s/it]
  GPU: 5417 MiB, 100 % | 5227 MiB, 100 % | RAM: 7/31 GB
[09:38:0